In [5]:
import glob
import os
import re
import pandas as pd

METRICS = ["Precision", "Recall", "F1-Score", "AUC", "MCC"]
GRANULARITIES = ["File", "Class", "Method", "Block"]
FILE_PREFIX = "within_project_"
FOLDER_PATH = "../results/trying_other_models/"


def clean_cell(text):
    """Removes LaTeX specific formatting elements."""
    text = re.sub(r"\\textbf\{([^}]+)\}", r"\1", text)
    text = re.sub(r"\\[a-zA-Z]+", "", text)
    return text.replace("\\", "").replace("%", "").strip()


def parse_file_metrics(filepath):
    """Parses a file, extracts all metrics across granularities, and normalizes values."""
    rows = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or "&" not in line:
                continue

            parts = [clean_cell(p) for p in line.split("&")]
            if len(parts) < 21:
                continue

            if parts[0].lower() == "average":
                continue

            project = parts[0]

            try:
                # Extract all 5 metrics for each of the 4 granularities
                for g_idx, granularity in enumerate(GRANULARITIES):
                    row_data = {
                        "Project": project,
                        "Granularity": granularity,
                    }

                    for m_idx, metric_name in enumerate(METRICS):
                        # Index calculation: offset of 1 for Project name + 5 metrics per granularity block
                        cell_idx = 1 + (g_idx * 5) + m_idx
                        raw_val = float(parts[cell_idx])

                        # Normalize to 0-100% scale
                        val = raw_val * 100 if 0 < raw_val <= 1.0 else raw_val
                        row_data[metric_name] = val

                    rows.append(row_data)

            except (ValueError, IndexError):
                continue

    return pd.DataFrame(rows)


def load_all_models(prefix="within_project_"):
    """Discovers files and extracts unified dataframes for all metrics."""
    compiled_frames = []
    search_path = os.path.join(FOLDER_PATH, f"{prefix}*")

    for filepath in glob.glob(search_path):
        filename = os.path.basename(filepath)
        model_name = filename.replace(prefix, "").replace(".txt", "")

        df = parse_file_metrics(filepath)
        if not df.empty:
            df["Model"] = model_name
            compiled_frames.append(df)

    if not compiled_frames:
        return pd.DataFrame()

    master_df = pd.concat(compiled_frames, ignore_index=True)

    # Rename original model for visualization consistency
    master_df["Model"] = master_df["Model"].replace(
        "ASMOTE_LightGBM", "LiteM (Original Model)"
    )

    return master_df

In [7]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def generate_interactive_plotly_dashboard(
    comparison_df, output_path="interactive_dashboard.html"
):
    """Generates an HTML Plotly dashboard combining:

    1. Aggregated Model Comparison Bar Plot (Top)
    2. Per-Project Granularity Line Charts (Bottom)
    Both updating dynamically via a single metric dropdown.
    """
    if comparison_df.empty:
        print("DataFrame is empty.")
        return

    df = comparison_df.copy()
    granularity_order = ["File", "Class", "Method", "Block"]
    unique_models = list(df["Model"].unique())
    metrics = [m for m in METRICS if m in df.columns]

    newer_models = [
        "LiteM (Original Model)",
        "ENN_ET",
        "ENN_LightGBM",
        "ENN_MLP",
        "ASMOTE_RF",
        "ASMOTE_XGB",
    ]
    old_school_models = [
        "ASMOTE_LogisticRegression",
        "ASMOTE_DecisionTree",
        "ASMOTE_NaiveBayes",
        "ASMOTE_Ridge",
        "ASMOTE_LDA",
    ]

    def categorize_generation(model):
        if model in newer_models:
            return "Newer Models"
        elif model in old_school_models:
            return "Old School Models"
        return "Other"

    df["Generation"] = df["Model"].apply(categorize_generation)

    color_palette = px.colors.qualitative.Alphabet
    model_colors = {
        model: color_palette[i % len(color_palette)]
        for i, model in enumerate(unique_models)
    }
    if "LiteM (Original Model)" in model_colors:
        model_colors["LiteM (Original Model)"] = "#000000"

    # Layout: Row 1 = Aggregated Bar Chart, Rows 2-3 = 2x2 Line Charts Grid
    fig = make_subplots(
        rows=3,
        cols=2,
        specs=[
            [{"colspan": 2}, None],
            [{}, {}],
            [{}, {}],
        ],
        subplot_titles=[
            "Overall Average Performance Across Granularities (95% CI)",
            "Granularity: File",
            "Granularity: Class",
            "Granularity: Method",
            "Granularity: Block",
        ],
        vertical_spacing=0.08,
        horizontal_spacing=0.06,
    )

    grid_map = {
        "File": (2, 1),
        "Class": (2, 2),
        "Method": (3, 1),
        "Block": (3, 2),
    }

    # Traces per metric block: 1 bar per model + (1 line per model * 4 granularities)
    traces_per_metric_block = len(unique_models) + (4 * len(unique_models))

    for m_idx, metric in enumerate(metrics):
        is_default = metric == "F1-Score"

        # -------------------------------------------------------------
        # 1. TOP SECTION: Aggregated Bar Plot Traces
        # -------------------------------------------------------------
        agg_df = (
            df.groupby(["Model", "Granularity"])[metric]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        agg_df["ci95"] = 1.96 * (agg_df["std"] / np.sqrt(agg_df["count"]))

        for model in unique_models:
            model_agg = agg_df[agg_df["Model"] == model]
            model_agg["Granularity"] = pd.Categorical(
                model_agg["Granularity"],
                categories=granularity_order,
                ordered=True,
            )
            model_agg = model_agg.sort_values("Granularity")

            fig.add_trace(
                go.Bar(
                    x=model_agg["Granularity"],
                    y=model_agg["mean"],
                    name=model,
                    legendgroup=model,
                    showlegend=is_default,
                    error_y=dict(
                        type="data", array=model_agg["ci95"], visible=True
                    ),
                    marker_color=model_colors[model],
                    visible=is_default,
                    hovertemplate=(
                        f"<b>Model:</b> {model}<br>"
                        "<b>Granularity:</b> %{x}<br>"
                        f"<b>Mean {metric}:</b> %{{y:.2f}}%<extra></extra>"
                    ),
                ),
                row=1,
                col=1,
            )

        # -------------------------------------------------------------
        # 2. BOTTOM SECTION: 2x2 Per-Project Line Charts Traces
        # -------------------------------------------------------------
        for granularity in granularity_order:
            row, col = grid_map[granularity]
            sub_df = df[df["Granularity"] == granularity]

            for model in unique_models:
                model_df = sub_df[sub_df["Model"] == model]
                if model_df.empty:
                    continue

                is_litem = model == "LiteM (Original Model)"
                gen = model_df["Generation"].iloc[0]
                dash_style = "dash" if gen == "Old School Models" else "solid"

                fig.add_trace(
                    go.Scatter(
                        x=model_df["Project"],
                        y=model_df[metric],
                        mode="lines+markers",
                        name=model,
                        legendgroup=model,
                        showlegend=False,
                        visible=is_default,
                        line=dict(
                            color=model_colors[model],
                            width=3.5 if is_litem else 1.5,
                            dash=dash_style,
                        ),
                        marker=dict(size=7 if is_litem else 5),
                        hovertemplate=(
                            f"<b>Model:</b> {model}<br>"
                            "<b>Project:</b> %{x}<br>"
                            f"<b>{metric}:</b> %{{y:.2f}}%<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=col,
                )

    # -----------------------------------------------------------------
    # BUILD SHARED METRIC DROPDOWN MENU
    # -----------------------------------------------------------------
    dropdown_buttons = []
    num_metrics = len(metrics)

    for i, metric in enumerate(metrics):
        visible_mask = [False] * (num_metrics * traces_per_metric_block)
        start_idx = i * traces_per_metric_block
        end_idx = start_idx + traces_per_metric_block
        visible_mask[start_idx:end_idx] = [True] * traces_per_metric_block

        dropdown_buttons.append(
            dict(
                label=metric,
                method="update",
                args=[
                    {"visible": visible_mask},
                    {
                        "title": f"Model Performance Dashboard Across Metrics ({metric})"
                    },
                ],
            )
        )

    fig.update_layout(
        barmode="group",
        updatemenus=[
            dict(
                active=metrics.index("F1-Score") if "F1-Score" in metrics else 0,
                buttons=dropdown_buttons,
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.0,
                xanchor="left",
                y=1.03,
                yanchor="top",
            )
        ],
        title={
            "text": "Model Performance Dashboard Across Metrics (F1-Score)",
            "x": 0.5,
            "xanchor": "center",
            "font": {"size": 20},
        },
        template="plotly_white",
        height=1400,
        legend=dict(
            title="Evaluated Models (Click to toggle)",
            orientation="h",
            y=-0.05,
            x=0.5,
            xanchor="center",
            font=dict(size=10),
        ),
        margin=dict(l=60, r=40, t=100, b=100),
    )

    fig.update_yaxes(title_text="Score (%)", range=[0, 105], row=1, col=1)
    fig.update_yaxes(title_text="Score (%)", range=[0, 105], row=2, col=1)
    fig.update_yaxes(title_text="Score (%)", range=[0, 105], row=3, col=1)

    fig.update_xaxes(tickangle=-45, row=3, col=1)
    fig.update_xaxes(tickangle=-45, row=3, col=2)

    fig.write_html(output_path, include_plotlyjs="cdn")
    print(f"Combined dashboard successfully saved to: {output_path}")


# Usage:
master_df = load_all_models()
generate_interactive_plotly_dashboard(master_df, "all_metrics_dashboard.html")

Combined dashboard successfully saved to: all_metrics_dashboard.html
